# OTA Update Prioritization - RL Warmup
This notebook contains a simplified environment to test the RL agent versus a FIFO baseline before connecting it to CARLA.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

## Simplified Environment Stub
We will model a list of pending updates with different 'Criticality' and 'Size' (Bandwidth usage).

In [ ]:
class OTAManagerEnv:
    def __init__(self, num_vehicles=5):
        self.num_vehicles = num_vehicles
        self.reset()
        
    def reset(self):
        # State: [Veh1_Crit, Veh1_Size, Veh2_Crit, Veh2_Size, ...]
        # Criticality 1-10, Size 1-100MB
        self.state = np.random.randint(1, 11, size=(self.num_vehicles, 2))
        # Keep a copy of the initial state for baseline comparisons
        self.initial_state = self.state.copy()
        self.done_mask = np.zeros(self.num_vehicles, dtype=bool)
        return self.state.flatten()
    
    def step(self, vehicle_index):
        if self.done_mask[vehicle_index]:
            return self.state.flatten(), 0, np.all(self.done_mask), {"error": "Already updated"}
            
        # Simple reward: Criticality / Size
        crit = self.state[vehicle_index, 0]
        size = self.state[vehicle_index, 1]
        reward = crit / (size / 10)
        
        # Mark as done
        self.done_mask[vehicle_index] = True
        self.state[vehicle_index] = [0, 0]
        
        done = np.all(self.done_mask)
        return self.state.flatten(), reward, done, {}

print("Environment stub initialized.")

## Baseline 1: FIFO (First In First Out)
FIFO will simply process vehicles from index 0 to 4, regardless of criticality or size.

In [ ]:
def run_fifo_baseline(env):
    env.reset()
    total_reward = 0
    for i in range(env.num_vehicles):
        _, reward, _, _ = env.step(i)
        total_reward += reward
    return total_reward

print("FIFO Baseline function defined.")

## Baseline 2: Random Agent
Randomly picks an available vehicle to update.

In [ ]:
def run_random_baseline(env):
    env.reset()
    total_reward = 0
    available = list(range(env.num_vehicles))
    np.random.shuffle(available)
    
    for i in available:
        _, reward, _, _ = env.step(i)
        total_reward += reward
    return total_reward

print("Random Baseline function defined.")

## Simulation Loop & Performance Metrics
We will run 100 simulations and compare the average total reward.

In [ ]:
env = OTAManagerEnv(num_vehicles=5)
num_episodes = 100
fifo_rewards = []
random_rewards = []

for _ in range(num_episodes):
    # Note: reset is called inside the run functions
    fifo_rewards.append(run_fifo_baseline(env))
    random_rewards.append(run_random_baseline(env))

plt.figure(figsize=(10, 5))
plt.plot(fifo_rewards, label='FIFO')
plt.plot(random_rewards, label='Random', alpha=0.7)
plt.title("Reward Comparison: FIFO vs Random Agent")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.legend()
plt.show()

print(f"Avg FIFO Reward: {np.mean(fifo_rewards):.2f}")
print(f"Avg Random Reward: {np.mean(random_rewards):.2f}")